In [ ]:
import sys 
sys.path.append( "../")
sys.path.append( "../../")

from datetime import datetime 

from typing import Dict, List, Tuple, Optional, Any 
from pathlib import Path 
from load_semantics import load_semantics, load_idiom_rules 
from semantics.semantic_models import * 
 
from typing import Any, Dict
import json
import yaml
from get_llm_model import *
import pandas as pd, numpy as np
import duckdb
import pprint 

from runtime.catalog import Catalog 
from runtime.smart_data import SmartData
from runtime.smart_data_tools import SmartDataTools

In [ ]:
import sys 
sys.path.append( "../")
sys.path.append( "../../")

from datetime import datetime 

from typing import Dict, List, Tuple, Optional, Any 
from pathlib import Path 
from load_semantics import load_semantics, load_idiom_rules 
from semantics.semantic_models import * 
 
from typing import Any, Dict
import json
import yaml
from get_llm_model import *
import pandas as pd, numpy as np
import duckdb
import pprint 

from runtime.catalog import Catalog 
from runtime.smart_data import SmartData
from runtime.smart_data_tools import SmartDataTools

semantic_catalog, sql_idioms, semantic_context = load_semantics( Path("../semantics/") )
idiom_rules, idiom_context = load_idiom_rules( idiom = 'duckdb',path = Path("../semantics/idioms.json") )
known_table_models = { item.name: item for item in semantic_catalog.tables }


In [ ]:
inj = pd.read_csv("../datasets/IX5I_4P/injectors.csv")
inj['DATE'] = pd.to_datetime( inj['DATE'],dayfirst=True)
inj['DAY']   = inj['DATE'].dt.day
inj['MONTH'] = inj['DATE'].dt.month
inj['YEAR']  = inj['DATE'].dt.year

pinj = pd.read_csv("../datasets/IX5I_4P/producers.csv")
pinj['DATE'] = pd.to_datetime( pinj['DATE'],dayfirst=True)
pinj['DAY']   = pinj['DATE'].dt.day
pinj['MONTH'] = pinj['DATE'].dt.month
pinj['YEAR']  = pinj['DATE'].dt.year


df_dict = {'injectors': inj, 'producers':pinj }

In [ ]:

from typing import Iterable, Union

semantic_catalog, sql_idioms, semantic_context = load_semantics( Path("../semantics/") )
idiom_rules, idiom_context = load_idiom_rules( idiom = 'duckdb',path = Path("../semantics/idioms.json") )
 

known_table_models = { item.name: item for item in semantic_catalog.tables }


catalog = Catalog()
catalog.initialize_from_named_dataframes( df_dict, known_table_models)

df = pd.DataFrame( {'name':['a','b'], 'age':[1,2]})
catalog.register_table( catalog.dataframe_to_table_card(df, name='aaa',description='sddgf', kind='derived'))
print(catalog.snapshot)#( catalog['aaa', 'injectors' ] ))


In [ ]:
from langchain_core.tools import StructuredTool, Tool


inj = pd.read_csv("../datasets/IX5I_4P/injectors.csv")
inj['DATE'] = pd.to_datetime( inj['DATE'],dayfirst=True)
inj['DAY']   = inj['DATE'].dt.day
inj['MONTH'] = inj['DATE'].dt.month
inj['YEAR']  = inj['DATE'].dt.year

pinj = pd.read_csv("../datasets/IX5I_4P/producers.csv")
pinj['DATE'] = pd.to_datetime( pinj['DATE'],dayfirst=True)
pinj['DAY']   = pinj['DATE'].dt.day
pinj['MONTH'] = pinj['DATE'].dt.month
pinj['YEAR']  = pinj['DATE'].dt.year

locs= pd.read_csv("../datasets/IX5I_4P/locations.csv")

df_dict = {'injectors': inj, 'producers':pinj , 'locations': locs }
data = SmartData()
data.initialize_from_named_dataframes( df_dict, known_table_models)
#data.register_derived_table( df, "a derived table", 'Something created on the fly')
print(data.catalog_snapshot())# ['injectors'] ))



In [ ]:

smart_data_tools = SmartDataTools( data=data)
tools = smart_data_tools.get_tools()

tools 




In [ ]:
#Your job is to generate and execute sql queries over a database to answer user questions
#You must always use sql_materialize when executing sql queries 
#You must always use sql_materialize when executing sql queries 

#You are an expert SQL generator for {idiom} databases. 
system_prompt_template = """
You are an expert analyst of databases.  
Your job is answer user questions grounded in the information contained in the database
You will never retrieve the contents of a table. 
Your will either indicate that a table can be reused or will generate and execute a sql to store a new derivd table in the database. 

# Workflow:
- Always call catalog_snapshot first.
- Analyze the information inthe catalog and the user question
- Reason step-by-step and emit a PLAN to answer the question.
- You MUST ALWAYS emit the PLAN in plain text using the provided tool. This PLAN message must contain no tool calls. Only after the PLAN message is sent may you call sql_* tools.
- ONLY use sql_materialize to create and store new intermediate tables needed to answer the user query.
- You MUST NOT recompute intermediate tables if an equivalent derived table already exists.
- If a derived table already contains ALL columns required to answer the question,
   you MUST reuse it.

   Recompute only if:
   - Required columns are missing, OR
   - Filtering conditions differ, OR
   - Aggregation level differs.


   -When reusing a given table simply state that the table can be reused, show the table name and its description. 
   Example:
   ```
   Table <fill> can be reused. Description: <fill>

   - NEVER call sql_materialize when reusing a table 

# Rules for tables and columns
- Choose a name for tables created that reflects the table contents
Examples: 
- Example 1: yearly_aggregated_oil_producer_per_subzone
- Example 2: gas_oil_water_cummulated_volumes 

- You must choose names for new columns that reflect their meaning
Examples: 
- Example 1: well_rank_according_to_water_production
- Example 2: distance_producer_to_nearest_injector

# SQL generation rules:
- ALWAYS use {idiom} compliant SQL syntax when generating queries.
Examples:
{idiom_examples}

- Do NOT use markdown code blocks (e.g., ```sql). 
- Do NOT use prefixes or explanations.
- Do NOT end the query with a semicolon ';'.
- Example: SELECT COUNT(DISTINCT well_id) AS well_count FROM injectors

# Business context:
Active wells within a given timeframe: 
- producer: liquid production > 0 within the timeframe.
- injector: water injection > 0 within the timeframe.
- "Current date": refers to the MAX("DATE") in the dataset.

- Summarization of injection: high-level figures on current 
active injectors, the total injection volume in each of the last three months 
split by subzones. The summary must indicate the number of inactive 
injectors. 

"""

# Final response:
r ="""
Final response must be a JSON string indicating the name of the that are part of the response 
their description. 
Example:
```
{{'tables':[
{{'count_of_injectors' : 'total number of injectors'}},
{{ table_name : description}}
... 
]
}}
```
"""




def system_prompt_builder( prompt_template, idiom:str, idiom_rules:dict ):
    
    idiom_examples = "\n".join([f"- {k}: {v}" for k, v in idiom_rules.items()])

    prompt = prompt_template.format(
        idiom=idiom, 
        idiom_examples=idiom_examples
    ) 

    return prompt 

idiom = 'duckdb'
#catalog_tool_name = 
#materialize_name  = 
#emit_plan_name    = 

prompt = system_prompt_builder( system_prompt_template, idiom, idiom_rules )

print( prompt, 'prompt length (approx)' , len(prompt.split()) ) 

# First agent and golden questions 

In [ ]:
import sys 
sys.path.append( "../")
sys.path.append( "../../")

from datetime import datetime 

from typing import Dict, List, Tuple, Optional, Any 
from pathlib import Path 
from load_semantics import load_semantics, load_idiom_rules 
from semantics.semantic_models import * 
 
from typing import Any, Dict
import json
import yaml
from get_llm_model import *
import pandas as pd, numpy as np
import duckdb
import pprint 

from runtime.catalog import Catalog 
from runtime.smart_data import SmartData
from runtime.smart_data_tools import SmartDataTools

In [ ]:
inj = pd.read_csv("../datasets/IX5I_4P/injectors.csv")
inj['DATE'] = pd.to_datetime( inj['DATE'],dayfirst=True)
inj['DAY']   = inj['DATE'].dt.day
inj['MONTH'] = inj['DATE'].dt.month
inj['YEAR']  = inj['DATE'].dt.year

pinj = pd.read_csv("../datasets/IX5I_4P/producers.csv")
pinj['DATE'] = pd.to_datetime( pinj['DATE'],dayfirst=True)
pinj['DAY']   = pinj['DATE'].dt.day
pinj['MONTH'] = pinj['DATE'].dt.month
pinj['YEAR']  = pinj['DATE'].dt.year

locs= pd.read_csv("../datasets/IX5I_4P/locations.csv")

df_dict = {'injectors': inj, 'producers':pinj , 'locations': locs }
data = SmartData()
data.initialize_from_named_dataframes( df_dict, known_table_models)
#data.register_derived_table( df, "a derived table", 'Something created on the fly')
#print(data.catalog_snapshot())# ['injectors'] ))


smart_data_tools = SmartDataTools( data=data)
tools = smart_data_tools.get_tools()
print(tools[0].func())


In [ ]:
def run_agent(agent, messages):
    config = {"configurable": {"thread_id": 123}}
    last_tool = None  
    for chunk in agent.stream(messages, config,  stream_mode="updates"):
        for step, response in chunk.items():

            print(f"\n--- {step.upper()} ---")

            msg = response["messages"][-1]
 

            # MODEL STEP (may contain tool calls)
            if step == "model":
                # Check if the model wants to call tools
                if hasattr(msg, "tool_calls") and msg.tool_calls:
                    for tool_call in msg.tool_calls:
                        print(f"TOOL CALL: {tool_call['name']}")
                        pprint.pprint(f"ARGUMENTS: {tool_call['args']}")
                else:
                    # Otherwise, show first 100 chars of text
                    content = msg.content or ""
                    print(f"RESPONSE: {content[:600]}...")
                    
            # 2. TOOLS STEP (Results)
            elif step == "tools":
                # msg here is a ToolMessage
                tool_name = getattr(msg, "name", "Unknown Tool")
                # Show first 600 chars of output to keep logs clean
                output = str(msg.content)
                if 'catalog' not in tool_name:
                    print(f"RESULT FROM [{tool_name}]: {output[:600]}...")
                
        

            # TOOL STEP (tool result)
            #elif step == "tools":
            #    if hasattr(msg, "content_blocks"):
            #        for block in msg.content_blocks:
            #            print("TOOL RESULT:",block["type"], type(block))
                        
            #            if 'catalog' not in last_tool:
                        
            #                if block["type"] == "text":
            #                    try:
            #                        parsed = json.loads(block["text"])
            #                        pprint(parsed[0:100])
            #                    except Exception as e:
            #                        print('*******error******')
            #                        print(block["text"], e )
            #                else:
            #                    print(block)
            #                    
            #            else:
            #                d = json.loads(block['text'])
            #                print('catalog keys', d['tables'].keys())
                pass#    else:
            #print(msg.content)

            # FINAL OUTPUT
            else:
                print("OTHER STEP:")
                print(msg.content)

            
    return response 




In [ ]:
query1 = "how many wells are there?"


query1_2 = "what proportion of those are injectors"


query2 = "What is the total water injection volume by year?"
query3 = "Tell me the total water injection volume for each subzone each year"
query4 = "rank wells by their variability (std) in water injection volume (the higher the grater the rank)?"
query4_1 = "whats the highest ranked well?"

query5 = "whats the frequency of observations in the dataset (D, M, Y) ?"
query6 = "summarize the injection data"
query7 = "Which well had the single highest WATER_INJECTION_VOLUME reading at any point in time and what was that reading?"
query8 = "What is the average monthly injection volume per well grouped by NAME and MONTH?"
query9 = """For each SUBZONE compute the year-over-year percentage change in total injection 
volume and report the largest drop
"""

queries = [
    #(query1, lambda x: int(x.loc[0,:].values[0]) == 5, 1),
    

    #(query2, lambda x: abs(float(x.loc[ x["YEAR"] == 2016, :].values[0][1]) - 56202.305) < 0.01, 2),
    #(query3, lambda x: abs(1306.96 - float(x.set_index(["YEAR", "SUBZONE"]).iloc[:, 0].loc[(2019, "WARA1")])) < 0.01, 2),
    (query4, lambda x: (x.loc[x["NAME"] == "I1", x.columns[-1]] == 1).any(), 2),
    #(query4_1, lambda x: (x.loc[x["NAME"] == "I1", x.columns[-1]] == 1).any(), 2),
    
    #(query7, lambda x: (x.shape[0] == 1 and (x.iloc[0]["NAME"] == "I1") and abs(float(x.iloc[0]["WATER_INJECTION_VOLUME"]) - 3537.0) < 0.1), 1),
    #(query8, lambda x: False, 2),
    #(query9, lambda x: False, 3),
]

In [ ]:
llm = azure_llm_if()

agent = create_agent(
        model=llm,
        system_prompt=prompt,
        tools=tools,
        #response_format=structured_output,
        #checkpointer= MemorySaver() 
    )

In [ ]:
#start = time.perf_counter()
#response = run_agent(agent, messages ) 
#end = time.perf_counter()

for n,item in enumerate(queries):

    print("\n\n",25*'=',n,25*'=',)
    user_query = item[0]
    print( user_query )
    messages = {"messages": [{"role": "user", "content": user_query}]}
    response = run_agent(agent, messages ) 
    print(50*'=',sep="\n\n")


In [ ]:
print( data.catalog_snapshot( 'well_injection_variability_rank' ))


In [ ]:
data.get_table_as_df('well_injection_variability_rank')#.sort_values(['NAME','MONTH'] )# .catalog_snapshot(['average_monthly_injection_per_well'])

In [ ]:
inj.groupby(['YEAR'])['WATER_INJECTION_VOLUME'].sum()

In [ ]:

system_prompt_template = """
You are an expert SQL generator for {idiom} based on the 
following database schema and description:

# Tables:
{context_lines}   

# Rules:
- Generate ONLY the SQL instruction. 
- Do NOT use markdown code blocks (e.g., ```sql). 
- Do NOT use prefixes or explanations.
- Do NOT end the query with a semicolon ';'.
- Example: SELECT COUNT(DISTINCT well_id) AS well_count FROM injectors

ALWAYS use {idiom} compliant SQL syntax.
Examples:
{idiom_examples}

 
# Business context:
Active wells within a given timeframe: 
- producer: liquid production > 0 within the timeframe.
- injector: water injection > 0 within the timeframe.

"Current date" refers to the MAX("DATE") in the dataset.

Summarization of injection: high-level figures on current 
active injectors, the total injection volume in each of the last three months 
split by subzones. The summary must indicate the number of inactive 
injectors. 

"""


idiom_name = "duckdb"
idiom_examples = idiom_context# "\n".join([f"- {k}: {v}" for k, v in idioms[idiom_name].items()])

context_dict = semantic_catalog.tables[0].model_dump(exclude_none=True)
context_yaml = yaml.dump(context_dict, sort_keys=False)



prompt = prompt_template.format(
    context_lines=context_yaml, 
    idiom=idiom_name, 
    idiom_examples=idiom_examples
) 

# response = llm.invoke(prompt)
print(prompt)

In [ ]:
idiom_examples

In [ ]:
PROMPT="""

You are an analytical SQL agent.
Your job is to generate and execute sql queries over a database to answer user questions
You must always use sql_materialize when executing sql queries 


"""

In [ ]:
idiom_context

In [ ]:
ANALYST_SYSTEM_PROMPT = """
You are an analytical SQL agent.
Your job is to generate and execute sql queries over a database to answer user questions
You must always use sql_materialize when executing sql queries 
===============================================================================
Workflow:
===============================================================================
You must:
1. Always call catalog_snapshot first.
2. Based on the information stored in the catalog, analyze the question, reason step-by-step and generate a PLAN to answer the question.
3. After catalog_snapshot, you MUST emit the PLAN in plain text. This PLAN message must contain no tool calls. Only after the PLAN message is sent may you call sql_* tools.
4. Use sql_materialize to create intermediate tables.
5. Final output rules:
   - If the result is a TABLE (multiple rows and/or columns), you MUST materialize it using sql_materialize.
   - If the result is a SINGLE VALUE (e.g., COUNT, MAX, MIN, AVG), DO NOT materialize a table. Return the result as a scalar answer.

Important:
Choose a name for intermediated tables created that reflects the table contents
Examples: 
Example 1: yearly_aggregated_oil_producer_per_subzone
Example 2: gas_oil_water_cummulated_volumes 

You must choose names for new columns that reflect their meaning
Examples: 
Example 1: well_rank_according_to_water_production
Example 2: distance_producer_to_nearest_injector

===============================================================================
Important: 
===============================================================================
- Do NOT produce narrative answers.
- Do NOT summarize.
- Never manually format rows.
- Never describe results in text.
- Never write CREATE or DROP in SQL.
- Never invent tables or columns.
- Scalar answers (single values) must be returned directly, not as tables.
- Only materialize results when they are genuinely tabular.

===============================================================================
MANDATORY REUSE RULE:
===============================================================================

After calling catalog_snapshot:

1. If a derived table already contains ALL columns required to answer the question,
   you MUST reuse it.

2. You MUST NOT recompute intermediate tables if an equivalent derived table already exists.

3. Recompute only if:
   - Required columns are missing, OR
   - Filtering conditions differ, OR
   - Aggregation level differs.

4. You MUST explicitly explain why reuse is not possible before creating new tables.


Important:
- Always create intermediate tables before selecting from them.
- If a query depends on a table, ensure it has been materialized first.


===============================================================================
SQL ENGINE CONSTRAINTS (STRICT)
===============================================================================

The SQL engine is SQLite.

You MUST use SQLite-compatible syntax only.

Strict rules:

❌ NEVER use:

DATE_TRUNC
EXTRACT
DATE_PART
ILIKE
FULL OUTER JOIN
RIGHT JOIN
FILTER (...)


Important:
- Avoid nested aggregates\u2014never wrap MAX/MIN inside SUM/AVG/etc. Example: WITH current_year AS (SELECT EXTRACT(YEAR FROM MAX(\"DATE\")) AS year FROM injectors), yearly_totals AS (...), yoy AS (...) SELECT ... FROM ... WHERE YEAR = (SELECT year FROM current_year)
- Use CTEs to capture helper scalars (like current_year via MAX(\"DATE\")) before performing group aggregations. 
For example: 
WITH current_year AS (SELECT EXTRACT(YEAR FROM MAX(\"DATE\")) AS year FROM injectors),\n         yearly_totals AS (...),\n         yoy AS (...)\n    SELECT ... FROM ... WHERE YEAR = (SELECT year FROM current_year)
     
     

✅ For date aggregation:
Use strftime() for grouping by month or year.

Example:
Monthly: strftime('%Y-%m', DATE)

Year: strftime('%Y', DATE)
If DATE is stored as TEXT in format 'YYYY-MM-DD', SQLite can parse it directly.

✅ For conditional logic:
Use CASE WHEN

✅ For string matching:
Use LIKE (case-sensitive unless COLLATE NOCASE is specified)

✅ For division:
Guard against division by zero using:

CASE WHEN denominator = 0 THEN NULL 
     ELSE numerator * 1.0 / denominator 
END
"""

In [ ]:

def build_prompt( user_query:str):
    

In [ ]:

def sanitize_df(df):

    df = df.copy()

    # Ensure index is not problematic
    if df.index.name is not None or not isinstance(df.index, pd.RangeIndex):
        df = df.reset_index()

    # Attempt to convert object columns
    for col in df.columns:
        if df[col].dtype == "object":
            # try datetime
            converted = pd.to_datetime(df[col], errors="ignore")
            if not pd.api.types.is_object_dtype(converted):
                df[col] = converted
                continue

            # try numeric
            converted = pd.to_numeric(df[col], errors="ignore")
            if not pd.api.types.is_object_dtype(converted):
                df[col] = converted

    return df

def register_table(conn, name: str, df, registry: set, columns: Dict[str, Any]):
    df = sanitize_df(df)
    conn.register(name, df)


con = duckdb.connect()
con.register("injectors", sanitize_df(inj))

In [ ]:

con = duckdb.connect()
con.register("injectors", sanitize_df(inj))


In [ ]:
from load_semantics import load_semantics, load_idiom_rules 
semantic_catalog, sql_idioms, semantic_context = load_semantics( Path("../semantics/") )
rules, idiom_context = load_idiom_rules( idiom = 'duckdb',path = Path("../semantics/idioms.json") )
 

In [ ]:

query = 'fdfgdfgdg'
prompt_template = """
You are an expert SQL generator for {idiom} based on the 
following database schema and description:

# Tables:
{context_lines}   

# Rules:
- Generate ONLY the SQL instruction. 
- Do NOT use markdown code blocks (e.g., ```sql). 
- Do NOT use prefixes or explanations.
- Do NOT end the query with a semicolon ';'.
- Example: SELECT COUNT(DISTINCT well_id) AS well_count FROM injectors

ALWAYS use {idiom} compliant SQL syntax.
Examples:
{idiom_examples}

 
# Business context:
Active wells within a given timeframe: 
- producer: liquid production > 0 within the timeframe.
- injector: water injection > 0 within the timeframe.

"Current date" refers to the MAX("DATE") in the dataset.

Summarization of injection: high-level figures on current 
active injectors, the total injection volume in each of the last three months 
split by subzones. The summary must indicate the number of inactive 
injectors. 

# Task: 
{user_query}

# {idiom} SQL:
"""


idiom_name = "duckdb"
idiom_examples = idiom_context# "\n".join([f"- {k}: {v}" for k, v in idioms[idiom_name].items()])

context_dict = semantic_catalog.tables[0].model_dump(exclude_none=True)
context_yaml = yaml.dump(context_dict, sort_keys=False)


# FIX: Changed 'idioms' to 'idiom' to match the template placeholder
prompt = prompt_template.format(
    user_query=query, 
    context_lines=context_yaml, 
    idiom=idiom_name, 
    idiom_examples=idiom_examples
) 

# response = llm.invoke(prompt)
print(prompt)

In [ ]:
query1 = "how many wells are there?"
query2 = "What is the total water injection volume by year?"
query3 = "Tell me the mean yearly water injection volume for each subzone"
query4 = "rank wells by their variability (std) in water injection volume (the higher the grater the rank)?"
query5 = "whats the frequency of observations in the dataset (D, M, Y) ?"
query6 = "summarize the injection data"
query7 = "Which well had the single highest WATER_INJECTION_VOLUME reading at any point in time and what was that reading?"
query8 = "What is the average monthly injection volume per well grouped by NAME and MONTH?"
query9 = """For each SUBZONE compute the year-over-year percentage change in total injection 
volume and report the largest drop
"""


queries = [
    #(query1, lambda x: int(x.loc[0,:].values[0]) == 5, 1),
    #(query2, lambda x: abs(float(x.loc[ x["YEAR"] == 2016, :].values[0][1]) - 56202.305) < 0.01, 2),
    #(query3, lambda x: abs(1306.96 - float(x.set_index(["YEAR", "SUBZONE"]).iloc[:, 0].loc[(2019, "WARA1")])) < 0.01, 2),
    #(query4, lambda x: (x.loc[x["NAME"] == "I1", x.columns[-1]] == 1).any(), 2),
    #(query7, lambda x: (x.shape[0] == 1 and (x.iloc[0]["NAME"] == "I1") and abs(float(x.iloc[0]["WATER_INJECTION_VOLUME"]) - 3537.0) < 0.1), 1),
    (query8, lambda x: False, 2),
    #(query9, lambda x: False, 3),
]

for n, query_item in enumerate(queries):
    query, checking_fn, complexity = query_item
    print(60 * "=")
    print(f"Complexity index: {complexity}")
    print(query)
   
    prompt = prompt_template.format(
        user_query=query, 
        context_lines=context_yaml, 
        idiom=idiom_name, 
        idiom_examples=idiom_examples
    )
    response = llm.invoke(prompt)

    print(response)

    # execute the generated SQL
    sql = response.content.strip()
    print(sql)

    try:
        result = con.execute(sql).fetchdf()
        display(result.sample(min(3, result.shape[0])))

        # check result
        print("success", checking_fn(result))
    except Exception as e:
        print("error", e)


In [ ]:
from pprint import pprint 
pprint(response.usage_metadata)
pprint(response.response_metadata["token_usage"]) 

# Now lets make it agentic and add some sort of structured output 

## Generate a prompt_builder that RAGs the questions, reasoning and sql 
## Add custom tools 


In [ ]:
response.response_metadata['token_usage']['prompt_tokens']

In [ ]:


class SmartData:
    def __init__(self, llm=None):
        self.con = duckdb.connect()
        self.tables = set()
        self.semantic = {}
        self.semantic_model = load_semantic_model()
        self.columns = {}
        self._llm = llm

    # -----------------------------
    # LLM PROPERTY
    # -----------------------------
    @property
    def llm(self):
        return self._llm

    @llm.setter
    def llm(self, model):
        self._llm = model

    # -----------------------------
    # CORE EXECUTION
    # -----------------------------
    def execute_query(self, sql: str):
        return self.con.execute(sql).fetchdf()

    # -----------------------------
    # TABLE REGISTRATION
    # -----------------------------
    def register_tables(self, tables: Dict[str, Any]):
        register_tables(self.con, tables, self.tables, self.columns)

    def register_table(self, name: str, df):
        register_table(self.con, name, df, self.tables, self.columns)

    # -----------------------------
    # SEMANTIC REGISTRATION
    # -----------------------------
    def register_semantic(self, name: str, description: str):
        if name not in self.tables:
            raise ValueError(f"Table '{name}' is not registered")
        self.semantic[name] = description

    # -----------------------------
    # SQL GENERATION
    # -----------------------------
    def generate_sql(self, user_query: str, **kwargs) -> str:
        if self._llm is None:
            raise ValueError("LLM is not set")

        context_lines = []
        for table in self.tables:
            desc = self.semantic.get(table, "")
            cols = self.columns.get(table, [])
            context_lines.append(
                f"Table: {table}\nDescription: {desc}\nColumns: {', '.join(cols)}"
            )

        context = "\n\n".join(context_lines)

        prompt = f"""
You are an expert SQL generator for DuckDB.

Available tables:
{context}

User request:
{user_query}

Generate a valid DuckDB SQL query only.
"""

        return self._llm(prompt, **kwargs)


llm = azure_llm_if()
print( llm )

